## SGAN

In [7]:
# Importing our dependencies

import tensorflow as tf
from tensorflow.keras import layers, Model, mixed_precision
from tensorflow.keras.utils import to_categorical
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [8]:
# Environment Setup

mixed_precision.set_global_policy('mixed_float16')

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Using GPU: {gpus[0].name}")
else:
    print("No GPU detected, using CPU instead.")

Using GPU: /physical_device:GPU:0


In [11]:
# Balancing the Dataset

df = pd.read_csv("dataset_original.csv")

if "hash" in df.columns:
    df = df.drop(columns=["hash"])

df_majority = df[df['malware'] == 1]
df_minority = df[df['malware'] == 0]

df_majority_downsampled = df_majority.sample(n=len(df_minority), random_state=42)
df_balanced = pd.concat([df_majority_downsampled, df_minority]).sample(frac=1, random_state=42)

df = df_balanced
df

,t_0,t_1,t_2,t_3,t_4,t_5,t_6,t_7,t_8,t_9,...,t_91,t_92,t_93,t_94,t_95,t_96,t_97,t_98,t_99,malware
20716,82,240,117,240,117,240,117,240,117,240,...,89,133,208,89,260,141,293,208,256,1
16944,208,286,76,110,240,117,208,187,208,198,...,240,286,76,306,286,240,286,76,306,1
24982,82,240,117,240,117,240,117,240,117,172,...,260,141,260,141,260,141,260,141,260,1
1469,82,198,274,158,215,86,82,274,37,240,...,215,274,158,215,274,215,274,158,215,0
34709,111,140,111,81,111,140,111,81,111,240,...,20,34,215,20,215,20,215,111,140,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23038,240,117,260,117,40,117,209,260,40,117,...,172,117,172,117,172,117,172,117,172,0
613,286,110,172,240,117,240,117,240,117,106,...,215,114,215,117,261,106,144,297,71,0
2229,286,110,172,240,117,240,117,240,117,106,...,215,114,215,117,261,106,144,297,117,0
8836,172,117,172,117,172,117,198,208,260,274,...,60,81,260,172,117,260,257,25,240,0


In [12]:
# Extracting X & y

X = df.drop(columns=["malware"]).values.astype(np.float32)
y = df["malware"].astype(int).values

X = (X - X.min()) / (X.max() - X.min() + 1e-8)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

input_dim = X_train.shape[1]
print(f"Training on {X_train.shape[0]} samples | Input dim: {input_dim}")

Training on 1726 samples | Input dim: 100


In [13]:
# D-model

def build_discriminator(input_dim, num_classes=3):
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(128, activation='relu')(inp)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    return Model(inp, out, name='Discriminator')

In [14]:
# G-model

def build_generator(latent_dim, output_dim):
    inp = layers.Input(shape=(latent_dim,))
    x = layers.Dense(64, activation='relu')(inp)
    x = layers.Dense(128, activation='relu')(x)
    out = layers.Dense(output_dim, activation='sigmoid', dtype='float32')(x)
    return Model(inp, out, name='Generator')

In [15]:
# GAN assembly

latent_dim = 64
G = build_generator(latent_dim, input_dim)
D = build_discriminator(input_dim)

opt_D = tf.keras.optimizers.Adam(learning_rate=0.0002)
opt_G = tf.keras.optimizers.Adam(learning_rate=0.0002)

D.compile(loss='categorical_crossentropy', optimizer=opt_D, metrics=['accuracy'])

# Combined GAN

D.trainable = False
z = layers.Input(shape=(latent_dim,))
fake_sample = G(z)
fake_pred = D(fake_sample)
GAN = Model(z, fake_pred)
GAN.compile(loss='categorical_crossentropy', optimizer=opt_G)

In [16]:
# Training setup

batch_size = 128
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(1024).batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [23]:
# Train GAN

epochs = 2000
steps_per_epoch = X_train.shape[0] // batch_size
total_steps = steps_per_epoch * epochs

print(f"Starting training for {epochs} epochs ({total_steps} total steps)...")

step = 0
for epoch in range(epochs):
    for real_samples, real_labels in train_ds:
        step += 1

        # Train Discriminator
        real_labels_oh = tf.one_hot(real_labels, depth=3)

        z_noise = tf.random.normal((batch_size, latent_dim))
        fake_samples = G.predict_on_batch(z_noise)
        fake_labels_oh = tf.one_hot(tf.fill([batch_size], 2), depth=3)

        X_combined = tf.concat([real_samples, fake_samples], axis=0)
        y_combined = tf.concat([real_labels_oh, fake_labels_oh], axis=0)

        D.trainable = True
        D_loss, D_acc = D.train_on_batch(X_combined, y_combined)

        # Train Generator
        z_noise = tf.random.normal((batch_size, latent_dim))
        misleading_labels = tf.one_hot(
            tf.random.uniform([batch_size], 0, 2, dtype=tf.int32), 3
        )
        D.trainable = False
        G_loss = GAN.train_on_batch(z_noise, misleading_labels)

        if step % 50 == 0:
            print(
                f"Step {step}/{total_steps} | "
                f"D_loss: {D_loss:.4f}, D_acc: {D_acc:.4f} | G_loss: {G_loss:.4f}"
            )

print("Training complete.")

Starting training for 2000 epochs (26000 total steps)...
Step 50/26000 | D_loss: 0.6564, D_acc: 0.7432 | G_loss: 2.0074
Step 100/26000 | D_loss: 0.6564, D_acc: 0.7431 | G_loss: 2.0070
Step 150/26000 | D_loss: 0.6565, D_acc: 0.7431 | G_loss: 2.0067
Step 200/26000 | D_loss: 0.6566, D_acc: 0.7430 | G_loss: 2.0067
Step 250/26000 | D_loss: 0.6567, D_acc: 0.7430 | G_loss: 2.0065
Step 300/26000 | D_loss: 0.6568, D_acc: 0.7429 | G_loss: 2.0063
Step 350/26000 | D_loss: 0.6569, D_acc: 0.7429 | G_loss: 2.0061
Step 400/26000 | D_loss: 0.6570, D_acc: 0.7428 | G_loss: 2.0056
Step 450/26000 | D_loss: 0.6571, D_acc: 0.7428 | G_loss: 2.0054
Step 500/26000 | D_loss: 0.6572, D_acc: 0.7427 | G_loss: 2.0051
Step 550/26000 | D_loss: 0.6573, D_acc: 0.7426 | G_loss: 2.0046
Step 600/26000 | D_loss: 0.6574, D_acc: 0.7426 | G_loss: 2.0043
Step 650/26000 | D_loss: 0.6574, D_acc: 0.7425 | G_loss: 2.0039
Step 700/26000 | D_loss: 0.6575, D_acc: 0.7425 | G_loss: 2.0036
Step 750/26000 | D_loss: 0.6576, D_acc: 0.7424 |

In [24]:
D_eval = Model(D.input, D.output[:, :2])
preds = D_eval.predict(X_test, batch_size=256)
pred_labels = np.argmax(preds, axis=1)

print("\nClassification Report:")
print(f"Test: \n {classification_report(y_test, pred_labels, digits=4)}")
print(f"Train: \n {classification_report(y_train, np.argmax(D_eval.predict(X_train, batch_size=256),axis=1), digits=4)}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step

Classification Report:
Test: 
               precision    recall  f1-score   support

           0     0.6404    0.8657    0.7362       216
           1     0.7929    0.5139    0.6236       216

    accuracy                         0.6898       432
   macro avg     0.7166    0.6898    0.6799       432
weighted avg     0.7166    0.6898    0.6799       432

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
Train: 
               precision    recall  f1-score   support

           0     0.6658    0.9189    0.7722       863
           1     0.8692    0.5388    0.6652       863

    accuracy                         0.7289      1726
   macro avg     0.7675    0.7289    0.7187      1726
weighted avg     0.7675    0.7289    0.7187      1726



In [ ]:
# --- Past Results ---

# Discriminator:
#     inp = layers.Input(shape=(input_dim,))
#     x = layers.Dense(128, activation='relu')(inp)
#     x = layers.Dropout(0.3)(x)
#     x = layers.Dense(64, activation='relu')(x)
#     out = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
#     return Model(inp, out, name='Discriminator')


# Generator:
#     inp = layers.Input(shape=(latent_dim,))
#     x = layers.Dense(64, activation='relu')(inp)
#     x = layers.Dense(128, activation='relu')(x)
#     out = layers.Dense(output_dim, activation='sigmoid', dtype='float32')(x)
#     return Model(inp, out, name='Generator')

"""epochs = 100"""

# Classification Report:
# Test:
#                precision    recall  f1-score   support

#            0     0.7972    0.8009    0.7991       216
#            1     0.8000    0.7963    0.7981       216

#     accuracy                         0.7986       432
#    macro avg     0.7986    0.7986    0.7986       432
# weighted avg     0.7986    0.7986    0.7986       432

# Train:
#                precision    recall  f1-score   support

#            0     0.8256    0.8227    0.8241       863
#            1     0.8233    0.8262    0.8248       863

#     accuracy                         0.8244      1726
#    macro avg     0.8245    0.8244    0.8244      1726
# weighted avg     0.8245    0.8244    0.8244      1726

"""epochs = 1000"""

# Classification Report:
# Test:
#                precision    recall  f1-score   support

#            0     0.8061    0.7315    0.7670       216
#            1     0.7542    0.8241    0.7876       216

#     accuracy                         0.7778       432
#    macro avg     0.7802    0.7778    0.7773       432
# weighted avg     0.7802    0.7778    0.7773       432

# Train:
#                precision    recall  f1-score   support

#            0     0.8566    0.8169    0.8363       863
#            1     0.8250    0.8633    0.8437       863

#     accuracy                         0.8401      1726
#    macro avg     0.8408    0.8401    0.8400      1726
# weighted avg     0.8408    0.8401    0.8400      1726

"""epochs = 2000"""

# Classification Report:
# Test:
#                precision    recall  f1-score   support

#            0     0.6404    0.8657    0.7362       216
#            1     0.7929    0.5139    0.6236       216

#     accuracy                         0.6898       432
#    macro avg     0.7166    0.6898    0.6799       432
# weighted avg     0.7166    0.6898    0.6799       432

# Train:
#                precision    recall  f1-score   support

#            0     0.6658    0.9189    0.7722       863
#            1     0.8692    0.5388    0.6652       863

#     accuracy                         0.7289      1726
#    macro avg     0.7675    0.7289    0.7187      1726
# weighted avg     0.7675    0.7289    0.7187      1726